# Scherrer, Lynch et al. (2023), dandiset 000691: more (n2) null decoders for ed9b-octant-decoding

Paper. Scherrer JR, Lynch GF, Zhang JJ, Fee MS (2023). An optical design enabling lightweight
and large field-of-view head-mounted microscopes. *Nature Methods* 20, 546-549.
[doi:10.1038/s41592-023-01806-1](https://doi.org/10.1038/s41592-023-01806-1).

Claims. `claims/000691/10.1038-s41592-023-01806-1.yaml`, written from the full text of the
version of record and the paper's code (GitHub `FeeLab/Feescope-Paper-Code`, commit d596646). It
decides what is tested and how.

Dandiset. [DANDI:000691](https://dandiarchive.org/dandiset/000691/0.260827.1358), version
0.260827.1358, the only published version (the paper cites none); see the main notebook,
`reproduce_scherrer_2023.ipynb`, for the reasons and the full data check.

Why this notebook. The criterion of ed9b-octant-decoding judges p under two readings of a null decoder: (n1)
one fit per shuffle and (n2) the average matrix of 200 fits per shuffle. The 1,000 (n2) decoders it
asks for need 200,000 fits, more than a day on four cores (measured in the decoding notebook), and are not run. The first verification asked for
more of them, to show where they lie; this notebook runs 8, a number chosen here to keep the notebook well
within the time limit, beside the one in `decoding_octant.ipynb`. They cannot give p < 0.001 and decide no
outcome alone.

| Claim | Data | Status |
|---|---|---|
| ed9b-octant-decoding | dandiset | 8 more (n2) null decoders |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

import kiloscope as ks

pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
urls = ks.asset_urls()
f = ks.open_nwb(urls)
S = ks.load_session(f)
POS = ks.mask_positions(ks.read_masks(urls[ks.NWB_PATH], f))
track, B = ks.load_behaviour(urls, S)
run = B["running"]
X_run = ks.moving_max(S["traces"][B["sel"]])[run]
X_F = np.asfortranarray(X_run)         # columns contiguous, for fast shuffles
print(f"loaded and tracked in {time.time() - started:.0f} s")

## Data check

The full check of the data is in the main notebook. Here: what this notebook decodes.

In [ ]:
LABELS_QD = ks.quad_dir_labels(B)[run]
LABELS_OCT = B["octant"][run]
print(f"imaging frames within the tracked video: {len(B['sel'])}; running (speed above its median): {run.sum()}")
print(f"activity: moving maximum over 30 frames of {X_run.shape[1]} footprints; values {X_run.min():.3f} to {X_run.max():.3f}")
print("frames per quadrant and direction class:", np.bincount(LABELS_QD, minlength=8).tolist())
print("frames per octant:", np.bincount(LABELS_OCT, minlength=8).tolist())
print(f"footprints with a position (the code's peak pixel): {(~np.isnan(POS['peak']).any(1)).sum()} of {len(POS['peak'])}")

## ed9b-octant-decoding

> Position within the maze can be decoded with octant resolution from the activity, above
> decoders trained on shuffled data.

Criterion (part): eight octants, lambda 6.75e-8, the average matrix of 200 fits; the null as for
fig2-quadrant-direction-decoding, under (n1) and (n2). Reproduced when the real median accuracy
exceeds every null value and chance under both.

The real decoder is computed again here with the seed of the decoding notebook, so that its value
is printed beside the null decoders.

In [ ]:
LABELS = LABELS_OCT
LAMBDA = ks.LAMBDA_OTHER
t0 = time.time()
real = ks.decoding_matrix(X_run, LABELS, 8, LAMBDA, 200, np.random.default_rng(69130), n_jobs=4)
print(f"real decoder, 200 fits, in {time.time() - t0:.0f} s: median accuracy {real['median']:.3f} (chance 0.125)")
t0 = time.time()
null2 = []
for s_ in np.random.SeedSequence(69133).spawn(8):
    rng_s = np.random.default_rng(s_)
    null2.append(ks.decoding_matrix(ks.shuffle_columns(X_F, rng_s), LABELS, 8, LAMBDA, 200, rng_s, n_jobs=4)["median"])
null2 = np.array(null2)
print(f"8 (n2) null decoders, 200 fits each, in {time.time() - t0:.0f} s")
print("their median accuracies:", np.round(null2, 3).tolist())
print(f"range {null2.min():.3f} to {null2.max():.3f}, mean {null2.mean():.3f}, standard deviation {null2.std(ddof=1):.4f}; "
      f"at or above the real {real['median']:.3f}: {(null2 >= real['median']).sum()}")
print(f"the real value lies {(real['median'] - null2.mean()) / null2.std(ddof=1):.0f} standard deviations of these null values above their mean "
      f"(descriptive only; no distribution is assumed for an outcome)")

**Result: blocked (not_runnable).** The real decoder, computed again with the decoding notebook's
seed, has a median accuracy of 0.326. The eight (n2) null decoders run here give 0.109 to 0.149
(mean 0.125), none at or above it. Eight decoders cannot show p < 0.001, which needs the 1,000 of
the criterion, so the claim stays blocked; these values show where the (n2) null lies.

In [ ]:
print(f"total run time {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome (this attempt) |
|---|---|
| ed9b-octant-decoding | blocked: not runnable (see the decoding notebook) |

How the analysis differs from the paper's. As in the decoding notebook: all 2948 footprints, the
position tracked from the video, scikit-learn's SGD in place of MATLAB's solver. Only 8 of the
1,000 (n2) null decoders the criterion asks for are run.